# Acoustic Teleoperation Analysis
**Configurations:**
- `flex` — FlexFrame (BPSK)
- `janus_E` — JANUS, Band E (BW = 6500 Hz)

**Pipeline:** log → clean → match → metrics → statistical tests → comparative plots

## 0. Configuration

In [75]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import re
import csv
import io
import os
import warnings
from scipy import stats
from scipy.stats import mannwhitneyu, shapiro, kruskal
from itertools import combinations

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR    = "./"           # Root directory for input data
FIGURES_DIR = "./figures"    # Output directory for exported PDF figures

CONFIGS = {
    "Lab: FlexFrame\n(BPSK)": f"{BASE_DIR}/lab/flex",
    "Lab: JANUS Band-E\n(BW=6500 Hz)": f"{BASE_DIR}/lab/janus_E",
    "Piovego: JANUS Band-E\n(BW=6500 Hz)": f"{BASE_DIR}/piovego/janus_E",
}

MAX_FORCE      = 50       # Force on the config file
MAX_DELAY_MS   = 15000    # matching window [ms] — must exceed max expected latency
ALPHA          = 0.05     # significance level for statistical tests

COLORS = ["#2196F3", "#FF5722", "#F57222"]   # one per config

# ── PDF export setup ───────────────────────────────────────────────────────
# Requires kaleido: pip install kaleido
pio.kaleido.scope.default_format = "pdf"
os.makedirs(FIGURES_DIR, exist_ok=True)

def save_fig(fig, name: str, width: int = 800, height: int = 500):
    """Export a Plotly figure as PDF to FIGURES_DIR."""
    path = os.path.join(FIGURES_DIR, f"{name}.pdf")
    fig.write_image(path, width=width, height=height)
    print(f"  ✔ Saved: {path}")

print("Configuration loaded.")

Configuration loaded.


## 1. Preprocessing Functions

In [76]:
def log_to_df(log_path: str) -> pd.DataFrame:
    """Parse receiver .log file → DataFrame"""
    pattern = re.compile(r"ROS_TIME_NS=(\d+).*?values=([-\d,]+)")
    rows = []
    with open(log_path, "r") as f:
        for line in f:
            m = pattern.search(line)
            if m:
                ros_time_ns = int(m.group(1))
                time_ms = ros_time_ns // 1_000_000
                values = list(map(int, m.group(2).split(",")))
                rows.append([time_ms] + values)
    cols = ["time"] + [f"value_{i}" for i in range(1, len(rows[0]))]
    return pd.DataFrame(rows, columns=cols)


def clean_controller_df(df: pd.DataFrame) -> pd.DataFrame:
    """Drop trigger columns, truncate floats, remove consecutive duplicates."""
    df = df.drop(columns=[c for c in ["leftT", "rightT"] if c in df.columns])
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(0).astype(float).astype(int)
    sensor_cols = [c for c in df.columns if c != "epoch"]
    mask = df[sensor_cols].ne(df[sensor_cols].shift()).any(axis=1)
    return pd.concat([df.iloc[[0]], df[mask]]).drop_duplicates().reset_index(drop=True)


def match(controller_df: pd.DataFrame,
          receiver_df: pd.DataFrame,
          max_delay_ms: int = MAX_DELAY_MS) -> pd.DataFrame:
    """
    Match sent commands (controller) to received commands (receiver).
    Returns a DataFrame with columns:
      controller_time, command_time, status
    where status ∈ {matched, lost, noSended}

    NOTE: 'noSended' means a packet was received but could not be matched
    to any sent command within the time window.  Investigate these carefully;
    they may indicate clock drift, retransmissions, or logging artefacts.
    """
    f1 = controller_df.sort_values("epoch").reset_index(drop=True)
    f2 = receiver_df.sort_values("time").reset_index(drop=True)

    # Normalise receiver values to controller scale
    r = pd.DataFrame()
    r["time"]   = pd.to_numeric(f2["time"], errors="coerce")
    r["leftV"]  = -(pd.to_numeric(f2.iloc[:, 1], errors="coerce") / MAX_FORCE)
    r["leftH"]  =  (pd.to_numeric(f2.iloc[:, 2], errors="coerce") / MAX_FORCE)
    r["rightH"] =  (pd.to_numeric(f2.iloc[:, 3], errors="coerce") / MAX_FORCE)
    r["rightV"] =  (pd.to_numeric(f2.iloc[:, 4], errors="coerce") / MAX_FORCE)

    for col in ["epoch", "leftH", "leftV", "rightH", "rightV"]:
        f1[col] = pd.to_numeric(f1[col], errors="coerce")

    results, used = [], set()
    j = 0
    for i in range(len(f1)):
        row1 = f1.iloc[i]
        t_ctrl = row1["epoch"]
        matched = False
        while j < len(r) and r.iloc[j]["time"] < t_ctrl:
            j += 1
        k = j
        while k < len(r) and r.iloc[k]["time"] <= t_ctrl + max_delay_ms:
            if k not in used:
                row2 = r.iloc[k]
                if (np.isclose(row1["leftH"],  row2["leftH"]) and
                    np.isclose(row1["leftV"],  row2["leftV"]) and
                    np.isclose(row1["rightH"], row2["rightH"]) and
                    np.isclose(row1["rightV"], row2["rightV"])):
                    results.append({"controller_time": t_ctrl,
                                    "command_time": row2["time"],
                                    "status": "matched"})
                    used.add(k)
                    matched = True
                    break
            k += 1
        if not matched:
            results.append({"controller_time": t_ctrl,
                            "command_time": None,
                            "status": "lost"})

    for idx in range(len(r)):
        if idx not in used:
            results.append({"controller_time": None,
                            "command_time": r.iloc[idx]["time"],
                            "status": "noSended"})

    out = pd.DataFrame(results)
    out["sort_time"] = out["controller_time"].fillna(out["command_time"])
    out = out.sort_values("sort_time").drop(columns=["sort_time"]).reset_index(drop=True)
    return out


print("Functions defined.")

Functions defined.


## 2. Load and Process All Configurations

In [77]:
matched_data = {}   # label → matched DataFrame
raw_stats    = {}   # label → dict of counts

for label, base in CONFIGS.items():
    print(f"\n{'='*55}")
    print(f"Processing: {label.replace(chr(10), ' ')}")
    print('='*55)

    try:
        rec_df  = log_to_df(f"{base}/received_log.log")
        ctrl_df = pd.read_csv(f"{base}/controller_log.csv")
    except FileNotFoundError as e:
        print(f"  [SKIP] File not found: {e}")
        continue

    ctrl_clean = clean_controller_df(ctrl_df)
    matched    = match(ctrl_clean, rec_df)

    if "matched" in matched["status"].values:
        first_matched_idx = matched.index[matched["status"] == "matched"][0]
        matched = matched.loc[first_matched_idx + 1:].reset_index(drop=True)

    n_total    = len(matched[matched["status"].isin(["matched", "lost"])])
    n_matched  = len(matched[matched["status"] == "matched"])
    n_lost     = len(matched[matched["status"] == "lost"])
    n_nosended = len(matched[matched["status"] == "noSended"])

    print(f"  Sent commands  : {n_total}")
    print(f"  Matched        : {n_matched}")
    print(f"  Lost           : {n_lost}")
    print(f"  noSended       : {n_nosended}  ← investigate these")

    if n_nosended > 0:
        print(f"[WARNING]: {n_nosended} received packets could not be matched "
              f"to any sent command. Possible causes: clock drift artefact, "
              f"duplicate transmission, or logging error. Exclude from PDR "
              f"calculation until cause is identified.")

    if n_total < 30:
        print(f"[WARNING]: sample size {n_total} < 30. "
              f"Statistical power is limited.")

    matched_data[label] = matched
    raw_stats[label] = {
        "n_sent": n_total, "n_matched": n_matched,
        "n_lost": n_lost,  "n_nosended": n_nosended
    }

print("\n[OK] All configurations processed.")
import pandas as pd


Processing: Lab: FlexFrame (BPSK)
  Sent commands  : 144
  Matched        : 142
  Lost           : 2
  noSended       : 0  ← investigate these

Processing: Lab: JANUS Band-E (BW=6500 Hz)
  Sent commands  : 241
  Matched        : 241
  Lost           : 0
  noSended       : 0  ← investigate these

Processing: Piovego: JANUS Band-E (BW=6500 Hz)
  Sent commands  : 93
  Matched        : 87
  Lost           : 6
  noSended       : 0  ← investigate these

[OK] All configurations processed.


## 3. Compute Per-Configuration Metrics

In [78]:
metrics = {}   # label → dict of metrics

for label, df in matched_data.items():
    s = raw_stats[label]
    df_m = df[df["status"] == "matched"].copy()
    df_m["latency_ms"] = df_m["command_time"] - df_m["controller_time"]

    # --- Sanity check: negative latencies indicate clock issue ---
    neg = (df_m["latency_ms"] < 0).sum()
    if neg > 0:
        print(f"[{label.replace(chr(10),' ')}] {neg} negative latencies detected. "
              f"Check clock synchronisation.")

    lat = df_m["latency_ms"].dropna()

    # PDR
    pdr = s["n_matched"] / s["n_sent"] if s["n_sent"] > 0 else np.nan

    # Jitter (RFC 3393 definition: mean absolute consecutive difference)
    jitter = df_m["latency_ms"].diff().abs().dropna()

    # Normality test (Shapiro-Wilk, valid for n < 5000)
    if len(lat) >= 3:
        sw_stat, sw_p = shapiro(lat)
        normal = sw_p > ALPHA
    else:
        sw_stat, sw_p, normal = np.nan, np.nan, False

    # CI: use t-distribution if normal & n>=30, else bootstrap
    n = len(lat)
    mean_lat = lat.mean()
    std_lat  = lat.std(ddof=1)

    if normal and n >= 30:
        se = std_lat / np.sqrt(n)
        ci_low  = mean_lat - 1.96 * se
        ci_high = mean_lat + 1.96 * se
        ci_method = "Normal (z=1.96)"
    else:
        # Bootstrap 95% CI
        rng = np.random.default_rng(42)
        boots = [rng.choice(lat, size=n, replace=True).mean() for _ in range(5000)]
        ci_low, ci_high = np.percentile(boots, [2.5, 97.5])
        ci_method = "Bootstrap (n=5000)"

    metrics[label] = {
        "pdr":          pdr,
        "packet_loss":  1.0 - pdr,
        "n_sent":       s["n_sent"],
        "n_matched":    s["n_matched"],
        "n_lost":       s["n_lost"],
        "lat_mean":     mean_lat,
        "lat_median":   lat.median(),
        "lat_std":      std_lat,
        "lat_p5":       lat.quantile(0.05),
        "lat_p95":      lat.quantile(0.95),
        "lat_min":      lat.min(),
        "lat_max":      lat.max(),
        "ci_low":       ci_low,
        "ci_high":      ci_high,
        "ci_method":    ci_method,
        "jitter_mean":  jitter.mean(),
        "jitter_std":   jitter.std(ddof=1),
        "jitter_max":   jitter.max(),
        "shapiro_p":    sw_p,
        "is_normal":    normal,
        "latency_vals": lat.values,
        "jitter_vals":  jitter.values,
    }

print("Metrics computed.")

Metrics computed.


## 4. Summary Table

In [79]:
rows = []
for label, m in metrics.items():
    rows.append({
        "Config":            label.replace("\n", " "),
        "N sent":            m["n_sent"],
        "PDR (%)": f"{m['pdr']*100:.1f}",
        "Loss (%)": f"{m['packet_loss']*100:.1f}",
        "Lat mean (ms)": f"{m['lat_mean']:.1f}",
        "Lat median (ms)": f"{m['lat_median']:.1f}",
        "Lat std (ms)": f"{m['lat_std']:.1f}",
        "Lat p5 (ms)": f"{m['lat_p5']:.1f}",
        "Lat p95 (ms)": f"{m['lat_p95']:.1f}",
        "CI 95%": f"[{m['ci_low']:.1f}, {m['ci_high']:.1f}]",
        "CI method":         m["ci_method"],
        "Jitter mean (ms)": f"{m['jitter_mean']:.1f}",
        "Jitter max (ms)": f"{m['jitter_max']:.1f}",
        "Normal (SW p)": f"{m['shapiro_p']:.4f}" if not np.isnan(m['shapiro_p']) else "N/A",
    })

summary = pd.DataFrame(rows).set_index("Config")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
display(summary.T)

Config,Lab: FlexFrame (BPSK),Lab: JANUS Band-E (BW=6500 Hz),Piovego: JANUS Band-E (BW=6500 Hz)
N sent,144,241,93
PDR (%),98.6,100.0,93.5
Loss (%),1.4,0.0,6.5
Lat mean (ms),1197.6,2738.8,3913.4
Lat median (ms),1198.5,2727.0,3616.0
Lat std (ms),73.9,142.0,1026.5
Lat p5 (ms),1090.0,2632.0,3041.6
Lat p95 (ms),1306.9,2833.0,5276.9
CI 95%,"[1185.4, 1209.3]","[2724.4, 2759.5]","[3715.1, 4147.3]"
CI method,Bootstrap (n=5000),Bootstrap (n=5000),Bootstrap (n=5000)


## 5. Statistical Tests

**Decision rule:** If any distribution is non-normal (Shapiro-Wilk p ≤ 0.05), use **Mann–Whitney U** (non-parametric) for pairwise comparison. If all are normal, use **Welch's t-test**. For global comparison across all 3, use **Kruskal–Wallis**.

In [80]:
labels = list(metrics.keys())
lat_arrays = [metrics[l]["latency_vals"] for l in labels]
all_normal = all(metrics[l]["is_normal"] for l in labels)

print("Normality check (Shapiro-Wilk):")
for l in labels:
    flag = "✅ normal" if metrics[l]["is_normal"] else "❌ non-normal"
    print(f"  {l.replace(chr(10),' '):30s}  p={metrics[l]['shapiro_p']:.4f}  {flag}")

test_name = "Welch's t-test" if all_normal else "Mann–Whitney U"
print(f"\n→ Using: {test_name} for pairwise comparisons")

# Global test
if len(lat_arrays) >= 3:
    kw_stat, kw_p = kruskal(*lat_arrays)
    print(f"\nKruskal–Wallis test (all groups):  H={kw_stat:.3f},  p={kw_p:.4f}")
    if kw_p < ALPHA:
        print("  → At least one group differs significantly (p < 0.05)")
    else:
        print("  → No significant difference detected across groups")

# Pairwise
print(f"\nPairwise comparisons ({test_name}):")
print(f"{'Pair':50s}  {'Statistic':>12s}  {'p-value':>10s}  Result")
print("-" * 90)
for l1, l2 in combinations(labels, 2):
    a1 = metrics[l1]["latency_vals"]
    a2 = metrics[l2]["latency_vals"]
    if all_normal:
        stat, p = stats.ttest_ind(a1, a2, equal_var=False)
    else:
        stat, p = mannwhitneyu(a1, a2, alternative="two-sided")
    sig = "significant" if p < ALPHA else "not significant"
    pair = f"{l1.replace(chr(10),' ')} vs {l2.replace(chr(10),' ')}"
    print(f"{pair:50s}  {stat:12.3f}  {p:10.4f}  {sig}")

Normality check (Shapiro-Wilk):
  Lab: FlexFrame (BPSK)           p=0.0000  ❌ non-normal
  Lab: JANUS Band-E (BW=6500 Hz)  p=0.0000  ❌ non-normal
  Piovego: JANUS Band-E (BW=6500 Hz)  p=0.0000  ❌ non-normal

→ Using: Mann–Whitney U for pairwise comparisons

Kruskal–Wallis test (all groups):  H=388.124,  p=0.0000
  → At least one group differs significantly (p < 0.05)

Pairwise comparisons (Mann–Whitney U):
Pair                                                   Statistic     p-value  Result
------------------------------------------------------------------------------------------
Lab: FlexFrame (BPSK) vs Lab: JANUS Band-E (BW=6500 Hz)         0.000      0.0000  significant
Lab: FlexFrame (BPSK) vs Piovego: JANUS Band-E (BW=6500 Hz)         0.000      0.0000  significant
Lab: JANUS Band-E (BW=6500 Hz) vs Piovego: JANUS Band-E (BW=6500 Hz)        98.500      0.0000  significant


## 6. Plots

In [81]:
# ── 6.1 PDR and Packet Loss ──────────────────────────────────────────────────
short_labels = [l.replace("\n", "\n") for l in labels]
pdrs = [metrics[l]["pdr"] * 100 for l in labels]
loss = [metrics[l]["packet_loss"] * 100 for l in labels]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Packet Delivery Ratio", "Packet Loss Rate"),
    horizontal_spacing=0.12,
)

for col_idx, (values, yaxis_title) in enumerate(
    [(pdrs, "PDR (%)"), (loss, "Packet Loss (%)")], start=1
):
    for i, (label, val) in enumerate(zip(short_labels, values)):
        fig.add_trace(
            go.Bar(
                x=[label], y=[val],
                marker_color=COLORS[i],
                marker_line_color="black", marker_line_width=0.8,
                text=[f"{val:.1f}%"], textposition="outside",
                showlegend=False, name=label,
            ),
            row=1, col=col_idx,
        )
    fig.update_yaxes(title_text=yaxis_title, range=[0, 115], row=1, col=col_idx)

fig.update_layout(
    width=900, height=450,
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", size=12),
    margin=dict(l=60, r=30, t=60, b=60),
)
fig.update_xaxes(showgrid=False, linecolor="black", mirror=True)
fig.update_yaxes(gridcolor="#e5e5e5", linecolor="black", mirror=True)

save_fig(fig, "pdr_loss", width=900, height=450)
fig.show()

  ✔ Saved: ./figures/pdr_loss.pdf


In [82]:
# ── 6.2 Comparative Violin Plot — Latency ────────────────────────────────────
fig = go.Figure()

for i, l in enumerate(labels):
    m = metrics[l]
    sl = l.replace("\n", " ")
    # Violin
    fig.add_trace(go.Violin(
        y=m["latency_vals"], x0=short_labels[i],
        name=sl,
        fillcolor=COLORS[i], opacity=0.6,
        line_color="rgba(0,0,0,0.6)",
        box_visible=True,
        meanline_visible=False,
        points=False,
        showlegend=True,
    ))
    # Mean ± 95% CI as scatter
    fig.add_trace(go.Scatter(
        x=[short_labels[i]],
        y=[m["lat_mean"]],
        error_y=dict(
            type="data",
            symmetric=False,
            array=[m["ci_high"] - m["lat_mean"]],
            arrayminus=[m["lat_mean"] - m["ci_low"]],
            color="red", thickness=1.5, width=6,
        ),
        mode="markers",
        marker=dict(symbol="diamond", color="red", size=7),
        name="Mean ± 95% CI" if i == 0 else None,
        showlegend=(i == 0),
    ))

fig.update_layout(
    title="Latency Distribution per Configuration",
    yaxis_title="End-to-End Latency (ms)",
    width=800, height=500,
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", size=12),
    margin=dict(l=60, r=30, t=60, b=60),
    violingap=0.3, violingroupgap=0.1,
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)", bordercolor="black", borderwidth=1),
)
fig.update_xaxes(showgrid=False, linecolor="black", mirror=True)
fig.update_yaxes(gridcolor="#e5e5e5", linecolor="black", mirror=True)

save_fig(fig, "latency_violin")
fig.show()

  ✔ Saved: ./figures/latency_violin.pdf


In [83]:
# ── 6.3 Latency CDF per configuration ───────────────────────────────────────
fig = go.Figure()

for i, l in enumerate(labels):
    lat = np.sort(metrics[l]["latency_vals"])
    cdf = np.arange(1, len(lat) + 1) / len(lat)
    fig.add_trace(go.Scatter(
        x=lat, y=cdf,
        mode="lines",
        name=l.replace("\n", " "),
        line=dict(color=COLORS[i], width=2),
    ))

fig.update_layout(
    title="CDF of End-to-End Latency",
    xaxis_title="Latency (ms)",
    yaxis_title="Cumulative Probability",
    width=800, height=500,
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", size=12),
    margin=dict(l=60, r=30, t=60, b=60),
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)", bordercolor="black", borderwidth=1),
)
fig.update_xaxes(showgrid=False, linecolor="black", mirror=True)
fig.update_yaxes(gridcolor="#e5e5e5", linecolor="black", mirror=True, range=[0, 1.05])

save_fig(fig, "latency_cdf")
fig.show()

  ✔ Saved: ./figures/latency_cdf.pdf


In [84]:
# ── 6.4 Latency over time (per config, one subplot each) ────────────────────
fig = make_subplots(
    rows=len(labels), cols=1,
    shared_xaxes=False,
    subplot_titles=[l.replace("\n", " ") for l in labels],
    vertical_spacing=0.12,
)

for i, l in enumerate(labels, start=1):
    m   = metrics[l]
    lat = m["latency_vals"]
    x   = list(range(len(lat)))

    # 95% CI shaded band
    fig.add_trace(go.Scatter(
        x=x + x[::-1],
        y=[m["ci_high"]] * len(x) + [m["ci_low"]] * len(x),
        fill="toself",
        fillcolor="rgba(255,0,0,0.10)",
        line=dict(color="rgba(255,255,255,0)"),
        name="95% CI" if i == 1 else None,
        showlegend=(i == 1),
    ), row=i, col=1)

    # Latency trace
    fig.add_trace(go.Scatter(
        x=x, y=lat,
        mode="lines",
        line=dict(color=COLORS[i - 1], width=1),
        opacity=0.85,
        name=l.replace("\n", " "),
        showlegend=True,
    ), row=i, col=1)

    # Mean line
    fig.add_trace(go.Scatter(
        x=[0, len(lat) - 1], y=[m["lat_mean"], m["lat_mean"]],
        mode="lines",
        line=dict(color="red", dash="dash", width=1),
        name="Mean" if i == 1 else None,
        showlegend=(i == 1),
    ), row=i, col=1)

    # Median line
    fig.add_trace(go.Scatter(
        x=[0, len(lat) - 1], y=[m["lat_median"], m["lat_median"]],
        mode="lines",
        line=dict(color="orange", dash="dot", width=1),
        name="Median" if i == 1 else None,
        showlegend=(i == 1),
    ), row=i, col=1)

    fig.update_yaxes(title_text="Latency (ms)", gridcolor="#e5e5e5",
                     linecolor="black", mirror=True, row=i, col=1)

fig.update_xaxes(title_text="Sample index",
                 showgrid=False, linecolor="black", mirror=True,
                 row=len(labels), col=1)
fig.update_xaxes(showgrid=False, linecolor="black", mirror=True)

fig.update_layout(
    height=350 * len(labels), width=900,
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", size=11),
    margin=dict(l=60, r=30, t=60, b=60),
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)", bordercolor="black", borderwidth=1),
)

save_fig(fig, "latency_time", width=900, height=350 * len(labels))
fig.show()

  ✔ Saved: ./figures/latency_time.pdf


In [85]:
# ── 6.5 Jitter — Comparative Violin ─────────────────────────────────────────
fig = go.Figure()

for i, l in enumerate(labels):
    fig.add_trace(go.Violin(
        y=metrics[l]["jitter_vals"], x0=short_labels[i],
        name=l.replace("\n", " "),
        fillcolor=COLORS[i], opacity=0.6,
        line_color="rgba(0,0,0,0.6)",
        box_visible=True,
        meanline_visible=True,
        points=False,
        showlegend=True,
    ))

fig.update_layout(
    title="Jitter Distribution per Configuration",
    yaxis_title="Jitter (ms)",
    width=800, height=500,
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", size=12),
    margin=dict(l=60, r=30, t=60, b=60),
    violingap=0.3, violingroupgap=0.1,
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)", bordercolor="black", borderwidth=1),
)
fig.update_xaxes(showgrid=False, linecolor="black", mirror=True)
fig.update_yaxes(gridcolor="#e5e5e5", linecolor="black", mirror=True)

save_fig(fig, "jitter_violin")
fig.show()

  ✔ Saved: ./figures/jitter_violin.pdf


In [86]:
# ── 6.6 Mean Latency with CI — bar chart ────────────────────────────────────
means  = [metrics[l]["lat_mean"]   for l in labels]
ci_lo  = [metrics[l]["lat_mean"] - metrics[l]["ci_low"]  for l in labels]
ci_hi  = [metrics[l]["ci_high"]  - metrics[l]["lat_mean"] for l in labels]

fig = go.Figure()

for i, (l, mean, lo, hi) in enumerate(zip(labels, means, ci_lo, ci_hi)):
    fig.add_trace(go.Bar(
        x=[short_labels[i]], y=[mean],
        name=l.replace("\n", " "),
        marker_color=COLORS[i],
        marker_line_color="black", marker_line_width=0.8,
        error_y=dict(
            type="data",
            symmetric=False,
            array=[hi], arrayminus=[lo],
            color="black", thickness=1.5, width=6,
        ),
    ))
    # CI method annotation
    fig.add_annotation(
        x=short_labels[i],
        y=mean + hi + max(means) * 0.04,
        text=metrics[l]["ci_method"].split(" ")[0],
        showarrow=False,
        font=dict(size=9, color="gray"),
    )

fig.update_layout(
    title="Mean End-to-End Latency with 95% CI",
    yaxis_title="Mean Latency (ms)",
    width=800, height=500,
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", size=12),
    margin=dict(l=60, r=30, t=60, b=60),
    showlegend=False,
    bargap=0.4,
)
fig.update_xaxes(showgrid=False, linecolor="black", mirror=True)
fig.update_yaxes(gridcolor="#e5e5e5", linecolor="black", mirror=True)

save_fig(fig, "latency_mean_ci")
fig.show()

  ✔ Saved: ./figures/latency_mean_ci.pdf


## 7. Methodological Warnings

The cell below prints all active warnings relevant to scientific validity.

In [87]:
print("=" * 60)
print("METHODOLOGICAL WARNINGS")
print("=" * 60)

for l, m in metrics.items():
    tag = l.replace("\n", " ")
    s   = raw_stats[l]

    if s["n_sent"] < 30:
        print(f"[WARNING] [{tag}] n={s['n_sent']} < 30: limited statistical power.")

    if not m["is_normal"]:
        print(f"[WARNING] [{tag}] Non-normal latency distribution (SW p={m['shapiro_p']:.4f}). "
              f"CI computed via bootstrap. Use median for central tendency.")

    if s["n_nosended"] > 0:
        print(f"[WARNING] [{tag}] {s['n_nosended']} 'noSended' packets. "
              f"Investigate before reporting PDR.")

    if m["lat_min"] < 0:
        print(f"[ERROR] [{tag}] Negative latencies present. "
              f"Clock synchronisation error suspected.")

print("\nNote: Only one experimental run per condition. Results are "
      "not replicable in a statistical sense. Report as preliminary evidence.")

METHODOLOGICAL WARNINGS
[WARNING] [Lab: FlexFrame (BPSK)] Non-normal latency distribution (SW p=0.0000). CI computed via bootstrap. Use median for central tendency.
[WARNING] [Lab: JANUS Band-E (BW=6500 Hz)] Non-normal latency distribution (SW p=0.0000). CI computed via bootstrap. Use median for central tendency.
[WARNING] [Piovego: JANUS Band-E (BW=6500 Hz)] Non-normal latency distribution (SW p=0.0000). CI computed via bootstrap. Use median for central tendency.

Note: Only one experimental run per condition. Results are not replicable in a statistical sense. Report as preliminary evidence.
